# 13_Task8_Step2 — Bổ sung đặc trưng hình học cho TCN

**Mục tiêu:** thử thêm khoảng cách giữa các cặp khớp bàn tay để model ít phụ thuộc hơn vào góc xoay.

- Model gốc để đối chiếu: `wide45_lownoise` (126 đặc trưng/khung).
- Model mới: 126 đặc trưng gốc + 210 khoảng cách giữa 21 khớp + 210 vận tốc của khoảng cách = **546 đặc trưng/khung**.
- Train tối đa 100 epoch, early stopping patience 18 theo Decision(5).
- Checkpoint mới lưu riêng ở `Data_321/checkpoints_v2/task8_step2_distance/TCN.pt`. Không ghi đè model cũ.
- Cell 6 so sánh hai model trên test với góc xoay giả lập; cell 8 mở webcam và nhấn `m` để đổi model ngay khi thử.

**Chạy lần lượt cell 1 → 8.** Cần mở notebook bằng kernel `train-gpu` có CUDA và MediaPipe. Notebook này đọc dữ liệu/checkpoint trong `~/Downloads/Data_321`.

In [18]:
#cell 1: import thư viện, đọc manifest và dựng dữ liệu 126 đặc trưng như notebook 07
import math, time, collections
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "Chưa bật CUDA; notebook không train trên CPU."
dev = "cuda"
BASE = Path.home() / "Downloads" / "Data_321"
assert (BASE / "manifest.csv").exists(), f"Không thấy {BASE / 'manifest.csv'}"
m = pd.read_csv(BASE / "manifest.csv")

def load_base_features(f):
    lm = np.load(BASE / f)["lm"]
    lm = lm - lm[:, :1]  # cổ tay làm gốc
    scale = np.linalg.norm(lm[:, 9], axis=1)[:, None, None]
    lm = lm / scale       # chuẩn hóa theo cỡ bàn tay
    pos = lm.reshape(len(lm), 63)
    has = ~np.isnan(pos[:, 0])
    vel = np.vstack([np.zeros((1, 63), np.float32), np.diff(pos, axis=0)])
    return np.nan_to_num(np.hstack([pos, vel])).astype(np.float32), has

Tmax = int(m.khung.max())

def build(df):
    X = np.zeros((len(df), Tmax, 126), np.float32)
    M = np.zeros((len(df), Tmax), bool)
    for i, f in enumerate(df.path):
        x, h = load_base_features(f)
        X[i, :len(x)] = x
        M[i, :len(x)] = h
    return (torch.tensor(X, device=dev),
            torch.tensor(df.nhan.values, dtype=torch.long, device=dev),
            torch.tensor(M, device=dev))

meta = {t: m[m.tap == t].reset_index(drop=True) for t in ["train", "val", "test"]}
data = {t: build(meta[t]) for t in meta}
print("CUDA:", torch.cuda.get_device_name(0), "| Tmax:", Tmax)
for t, (X, y, M) in data.items():
    print(t, tuple(X.shape), "| video:", len(y), "| khung có tay:", int(M.sum().item()))

CKPT_OLD = BASE / "checkpoints_v2" / "wide45_lownoise" / "TCN.pt"
assert CKPT_OLD.exists(), f"Không thấy checkpoint baseline: {CKPT_OLD}"

CUDA: NVIDIA GeForce RTX 3050 Ti Laptop GPU | Tmax: 355
train (151, 355, 126) | video: 151 | khung có tay: 19108
val (31, 355, 126) | video: 31 | khung có tay: 3782
test (33, 355, 126) | video: 33 | khung có tay: 3930


In [19]:
#cell 2: tạo 210 khoảng cách khớp + 210 vận tốc khoảng cách, giữ nguyên 126 đặc trưng gốc
# 21 điểm có 21*20/2 = 210 cặp khác nhau
PAIR_I, PAIR_J = torch.triu_indices(21, 21, offset=1, device=dev)
N_DIST = len(PAIR_I)
IN_NEW = 126 + N_DIST + N_DIST
assert N_DIST == 210 and IN_NEW == 546

def features_from_pos(pos, hand_mask):
    # pos: (batch, time, 21, 3), đã chuẩn hóa; khung mất tay sẽ bị đặt về 0
    valid = hand_mask[:, :, None, None]
    pos = torch.where(valid, pos, torch.zeros_like(pos))
    b, T = pos.shape[:2]

    pos_flat = pos.reshape(b, T, 63)
    vel = torch.zeros_like(pos_flat)
    if T > 1:
        pair_valid = (hand_mask[:, 1:] & hand_mask[:, :-1])[:, :, None]
        vel[:, 1:] = (pos_flat[:, 1:] - pos_flat[:, :-1]) * pair_valid

    diff = pos[:, :, PAIR_I, :] - pos[:, :, PAIR_J, :]
    dist = torch.linalg.vector_norm(diff, dim=-1)  # (batch, time, 210)
    dist = dist * hand_mask[:, :, None]

    dist_vel = torch.zeros_like(dist)
    if T > 1:
        pair_valid = (hand_mask[:, 1:] & hand_mask[:, :-1])[:, :, None]
        dist_vel[:, 1:] = (dist[:, 1:] - dist[:, :-1]) * pair_valid

    return torch.cat([pos_flat, vel, dist, dist_vel], dim=-1)

def geo_from_base(X, hand_mask):
    # X: (batch, time, 126). Dùng phần tọa độ, tính lại vận tốc và đặc trưng hình học.
    pos = X[..., :63].reshape(X.shape[0], X.shape[1], 21, 3)
    return features_from_pos(pos, hand_mask)

geo_data = {}
for split, (X, y, M) in data.items():
    geo_data[split] = (geo_from_base(X, M), y, M)

print("Số đặc trưng mới:", IN_NEW)
print("Shape train mới:", tuple(geo_data["train"][0].shape))
assert geo_data["train"][0].shape[-1] == 546

Số đặc trưng mới: 546
Shape train mới: (151, 355, 546)


In [20]:
#cell 3: định nghĩa TCN tương thích với checkpoint cũ và model mới
class Block(nn.Module):
    def __init__(self, c, d):
        super().__init__()
        self.pad = 2 * d
        self.conv = nn.Conv1d(c, c, 3, dilation=d)
        self.drop = nn.Dropout(0.1)
    def forward(self, x):
        return x + self.drop(F.relu(self.conv(F.pad(x, (self.pad, 0)))))

class TCN(nn.Module):
    def __init__(self, in_features=126, c=64):
        super().__init__()
        self.inp = nn.Linear(in_features, c)
        self.blocks = nn.Sequential(*[Block(c, d) for d in (1, 2, 4, 8, 16)])
        self.out = nn.Linear(c, 3)
    def forward(self, x):
        return self.out(self.blocks(self.inp(x).transpose(1, 2)).transpose(1, 2))

old_ck = torch.load(CKPT_OLD, map_location=dev, weights_only=False)
net_old = TCN(in_features=126).to(dev)
net_old.load_state_dict(old_ck["w"])
net_old.eval()

CKPT_NEW = BASE / "checkpoints_v2" / "task8_step2_distance" / "TCN.pt"
CFG = {"feature_version": "pairwise_distances_v1", "rot": (40, 45, 40),
       "noise": 0.005, "epochs": 100, "patience": 18, "in_features": IN_NEW}
print(f"Baseline đã nạp: {CKPT_OLD} | epoch {old_ck.get('epoch', '?')}")
print("Checkpoint model mới:", CKPT_NEW)

Baseline đã nạp: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_v2\wide45_lownoise\TCN.pt | epoch 49
Checkpoint model mới: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_v2\task8_step2_distance\TCN.pt


In [21]:
#cell 4: augmentation 3D + lật gương + nhiễu; sau đó tính lại toàn bộ đặc trưng 546 chiều
def augment_geo(X, hand_mask, rot_deg=(40, 45, 40), noise_std=0.005):
    b, T = X.shape[:2]
    pos = X[..., :63].reshape(b, T, 21, 3).clone()
    valid = hand_mask[:, :, None, None]

    rx, ry, rz = [math.radians(d) for d in rot_deg]
    ax = (torch.rand(b, 1, 1, device=X.device) * 2 - 1) * rx
    ay = (torch.rand(b, 1, 1, device=X.device) * 2 - 1) * ry
    az = (torch.rand(b, 1, 1, device=X.device) * 2 - 1) * rz

    xx, yy, zz = pos[..., 0], pos[..., 1], pos[..., 2]
    y1, z1 = yy * ax.cos() - zz * ax.sin(), yy * ax.sin() + zz * ax.cos()
    x2, z2 = xx * ay.cos() + z1 * ay.sin(), -xx * ay.sin() + z1 * ay.cos()
    pos = torch.stack([x2 * az.cos() - y1 * az.sin(),
                       x2 * az.sin() + y1 * az.cos(), z2], dim=-1)

    flip = torch.rand(b, 1, 1, device=X.device) < 0.5
    pos[..., 0] = torch.where(flip, -pos[..., 0], pos[..., 0])
    pos = pos + torch.randn_like(pos) * noise_std * valid
    pos = torch.where(valid, pos, torch.zeros_like(pos))
    return features_from_pos(pos, hand_mask)

def loss_acc(net, X, y, M):
    out = net(X)
    loss_each = F.cross_entropy(out.transpose(1, 2), y[:, None].expand(-1, X.shape[1]), reduction="none")
    correct = out.argmax(-1) == y[:, None]
    return (loss_each * M).sum() / M.sum().clamp(min=1), (correct & M).sum() / M.sum().clamp(min=1)

# Kiểm tra khoảng cách không đổi khi xoay 3D: sai số chỉ nên ở mức số học rất nhỏ.
Xsample, _, Msample = data["train"]
pos_sample = Xsample[:2, :20, :63].reshape(2, 20, 21, 3)
d0 = torch.linalg.vector_norm(pos_sample[:, :, PAIR_I] - pos_sample[:, :, PAIR_J], dim=-1)
theta = math.radians(35)
p = pos_sample.clone()
xx, yy, zz = p[..., 0], p[..., 1], p[..., 2]
p = torch.stack([xx * math.cos(theta) - yy * math.sin(theta),
                 xx * math.sin(theta) + yy * math.cos(theta), zz], dim=-1)
d1 = torch.linalg.vector_norm(p[:, :, PAIR_I] - p[:, :, PAIR_J], dim=-1)
print("Sai số khoảng cách sau xoay 35°:", (d0 - d1).abs().max().item())

Sai số khoảng cách sau xoay 35°: 2.384185791015625e-07


In [22]:
#cell 5: train model mới trên CUDA; tối đa 100 epoch, dừng sớm sau 18 epoch không cải thiện
FORCE_RETRAIN = False  # True chỉ khi muốn train lại model Step 2

def train_step2(force=False, bs=16):
    net = TCN(in_features=IN_NEW).to(dev)
    if CKPT_NEW.exists() and not force:
        ck = torch.load(CKPT_NEW, map_location=dev, weights_only=False)
        if ck.get("cfg") == CFG:
            net.load_state_dict(ck["w"])
            print(f"Nạp checkpoint Step 2 có sẵn | epoch {ck['epoch']} | val_loss {ck['val_loss']:.4f}")
            return net.eval()
        print("Checkpoint Step 2 có cấu hình/phiên bản đặc trưng khác; sẽ train lại.")

    torch.manual_seed(0)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)
    Xt, yt, Mt = data["train"]
    Xv, yv, Mv = geo_data["val"]
    best, wait, hist = float("inf"), 0, []
    t0 = time.time()

    for ep in range(1, CFG["epochs"] + 1):
        net.train()
        perm = torch.randperm(len(Xt), device=dev)
        total = 0.0
        for start in range(0, len(Xt), bs):
            ids = perm[start:start + bs]
            xb = augment_geo(Xt[ids], Mt[ids], CFG["rot"], CFG["noise"])
            loss, _ = loss_acc(net, xb, yt[ids], Mt[ids])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            total += loss.item() * len(ids)

        net.eval()
        with torch.no_grad():
            vl, va = loss_acc(net, Xv, yv, Mv)
        row = {"epoch": ep, "train_loss": total / len(Xt),
               "val_loss": float(vl.item()), "val_acc": float(va.item())}
        hist.append(row)

        if row["val_loss"] < best:
            best, wait = row["val_loss"], 0
            CKPT_NEW.parent.mkdir(parents=True, exist_ok=True)
            torch.save({"w": net.state_dict(), "epoch": ep, "val_loss": best,
                        "hist": hist, "model": "TCN", "variant": "task8_step2_distance",
                        "cfg": CFG}, CKPT_NEW)
        else:
            wait += 1

        if ep % 5 == 0 or wait >= CFG["patience"] or ep == CFG["epochs"]:
            print(f"Step2 ep{ep:03d} | train {row['train_loss']:.4f} | "
                  f"val {row['val_loss']:.4f} | val_acc {row['val_acc']:.3f}")
        if wait >= CFG["patience"]:
            print("Dừng sớm: validation loss không giảm.")
            break

    ck = torch.load(CKPT_NEW, map_location=dev, weights_only=False)
    net.load_state_dict(ck["w"])
    print(f"Model mới tốt nhất epoch {ck['epoch']} | val_loss {ck['val_loss']:.4f}")
    print("Đã lưu checkpoint:", CKPT_NEW)
    print(f"Thời gian train: {time.time() - t0:.0f}s")
    return net.eval()

net_new = train_step2(force=FORCE_RETRAIN)

Nạp checkpoint Step 2 có sẵn | epoch 25 | val_loss 0.2104


In [23]:
#cell 6: so sánh baseline và Step 2 trên test ở góc 0/15/30/45/60 độ
Xte, yte, Mte = data["test"]
THR = 0.7

def rotate_base(X, hand_mask, ax_deg, ay_deg, az_deg):
    b, T = X.shape[:2]
    pos = X[..., :63].reshape(b, T, 21, 3).clone()
    valid = hand_mask[:, :, None, None]
    ax, ay, az = [torch.as_tensor(a, dtype=torch.float32, device=X.device) * math.pi / 180
                  for a in (ax_deg, ay_deg, az_deg)]
    xx, yy, zz = pos[..., 0], pos[..., 1], pos[..., 2]
    y1, z1 = yy * ax.cos() - zz * ax.sin(), yy * ax.sin() + zz * ax.cos()
    x2, z2 = xx * ay.cos() + z1 * ay.sin(), -xx * ay.sin() + z1 * ay.cos()
    pos = torch.stack([x2 * az.cos() - y1 * az.sin(),
                       x2 * az.sin() + y1 * az.cos(), z2], dim=-1)
    pos = torch.where(valid, pos, torch.zeros_like(pos))
    pos_flat = pos.reshape(b, T, 63)
    vel = torch.zeros_like(pos_flat)
    if T > 1:
        pair_valid = (hand_mask[:, 1:] & hand_mask[:, :-1])[:, :, None]
        vel[:, 1:] = (pos_flat[:, 1:] - pos_flat[:, :-1]) * pair_valid
    return torch.cat([pos_flat, vel], dim=-1)

def evaluate(net, Xin, labels, mask):
    with torch.no_grad():
        conf, pred = net(Xin).softmax(-1).max(-1)
    ok = pred == labels[:, None]
    selected = (conf >= THR) & mask
    n_all, n_sel = mask.sum().item(), selected.sum().item()
    return {"acc_all": (ok & mask).sum().item() / max(n_all, 1) * 100,
            "acc_conf": (ok & selected).sum().item() / max(n_sel, 1) * 100,
            "coverage": n_sel / max(n_all, 1) * 100}

MAGS, REPEAT = [0, 15, 30, 45, 60], 5
g_rand = torch.Generator(device=dev).manual_seed(0)
rows = []
for mag in MAGS:
    results = {"wide45_lownoise": [], "task8_step2_distance": []}
    for _ in range(1 if mag == 0 else REPEAT):
        if mag == 0:
            Xr = Xte
        else:
            rand_angle = lambda: (torch.rand(len(Xte), 1, 1, device=dev, generator=g_rand) * 2 - 1) * mag
            Xr = rotate_base(Xte, Mte, rand_angle(), rand_angle(), rand_angle())
        results["wide45_lownoise"].append(evaluate(net_old, Xr, yte, Mte))
        Xgeo = geo_from_base(Xr, Mte)
        results["task8_step2_distance"].append(evaluate(net_new, Xgeo, yte, Mte))
    for name, vals in results.items():
        rows.append({"model": name, "goc": mag,
                     **{k: float(np.mean([v[k] for v in vals])) for k in vals[0]}})

res = pd.DataFrame(rows)
for metric, title in [("acc_all", "Độ chính xác mọi khung có tay (%)"),
                      ("acc_conf", f"Độ chính xác khi tin cậy ≥ {THR} (%)"),
                      ("coverage", f"Độ phủ: % khung đạt ngưỡng {THR}")]:
    print("\n" + title)
    display(res.pivot_table(index="model", columns="goc", values=metric).round(1))

p = res.pivot_table(index="model", columns="goc", values="acc_all")
d0 = p.loc["task8_step2_distance", 0] - p.loc["wide45_lownoise", 0]
d45 = p.loc["task8_step2_distance", 45] - p.loc["wide45_lownoise", 45]
print(f"\nChênh lệch Step 2 so với baseline: 0° = {d0:+.1f} điểm; ±45° = {d45:+.1f} điểm.")
if d0 >= -1 and d45 > 0:
    print("KẾT QUẢ SỐ LIỆU: ĐẠT tiêu chí sơ bộ. Vẫn cần xác nhận bằng webcam.")
else:
    print("KẾT QUẢ SỐ LIỆU: CHƯA ĐẠT tiêu chí sơ bộ; giữ wide45_lownoise làm baseline.")


Độ chính xác mọi khung có tay (%)


goc,0,15,30,45,60
model,,,,,
task8_step2_distance,93.2,92.8,92.9,92.8,92.3
wide45_lownoise,92.5,92.3,91.3,91.2,90.4



Độ chính xác khi tin cậy ≥ 0.7 (%)


goc,0,15,30,45,60
model,,,,,
task8_step2_distance,95.8,95.9,95.9,95.3,95.6
wide45_lownoise,96.7,96.7,97.2,97.7,96.4



Độ phủ: % khung đạt ngưỡng 0.7


goc,0,15,30,45,60
model,,,,,
task8_step2_distance,91.8,91.7,91.7,92.1,91.8
wide45_lownoise,89.4,89.2,87.2,85.9,84.6



Chênh lệch Step 2 so với baseline: 0° = +0.6 điểm; ±45° = +1.5 điểm.
KẾT QUẢ SỐ LIỆU: ĐẠT tiêu chí sơ bộ. Vẫn cần xác nhận bằng webcam.


In [24]:
#cell 7: chuẩn bị MediaPipe và hàm dự đoán webcam cho cả baseline và Step 2
import cv2
import mediapipe as mp

TASK = BASE / "hand_landmarker.task"
assert TASK.exists(), f"Không thấy {TASK}"
CLASSES = ["paper", "rock", "scissors"]
COLORS = {0: (60, 60, 255), 1: (255, 150, 40), 2: (60, 210, 60)}
GRAY, BLACK = (190, 190, 190), (0, 0, 0)
BONES = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
         (5,9),(9,10),(10,11),(11,12),(9,13),(13,14),(14,15),(15,16),
         (13,17),(17,18),(18,19),(19,20),(0,17)]

landmarker = mp.tasks.vision.HandLandmarker.create_from_options(
    mp.tasks.vision.HandLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(TASK)),
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
        num_hands=1
    )
)
T0 = time.perf_counter()

def detect(frame_bgr, ts_ms):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    result = landmarker.detect_for_video(mp.Image(mp.ImageFormat.SRGB, rgb), ts_ms)
    if not result.hand_landmarks:
        return None
    return np.array([[p.x, p.y, p.z] for p in result.hand_landmarks[0]], np.float32)

FPS_MODEL, WIN = 120, 128
buf = collections.deque(maxlen=WIN)
state = {"prev_lm": None, "prev_t": None}

def push(lm, t):
    prev, pt = state["prev_lm"], state["prev_t"]
    n = 1 if pt is None else max(1, round((t - pt) * FPS_MODEL))
    if lm is None:
        buf.extend([np.full((21, 3), np.nan, np.float32)] * n)
    elif prev is None:
        buf.append(lm)
    else:
        buf.extend(prev + (lm - prev) * (k / n) for k in range(1, n + 1))
    state["prev_lm"], state["prev_t"] = lm, t

def webcam_base_features(lm):
    # lm: (T,21,3), có thể chứa NaN ở khung mất tay
    has = ~np.isnan(lm[:, 0, 0])
    q = lm - lm[:, :1]
    scale = np.linalg.norm(q[:, 9], axis=1)[:, None, None]
    q = np.nan_to_num(q / np.maximum(scale, 1e-8)).astype(np.float32)
    q[~has] = 0
    pos = q.reshape(len(q), 63)
    vel = np.zeros_like(pos)
    if len(pos) > 1:
        pair_valid = (has[1:] & has[:-1])[:, None]
        vel[1:] = (pos[1:] - pos[:-1]) * pair_valid
    return np.hstack([pos, vel]).astype(np.float32), has

def predict(which):
    lm = np.stack(buf)
    if np.isnan(lm[-1, 0, 0]):
        return None
    base, has = webcam_base_features(lm)
    xb = torch.from_numpy(base)[None].to(dev)
    mb = torch.from_numpy(has)[None].to(dev)
    with torch.no_grad():
        if which == "old":
            probs = net_old(xb).softmax(-1)[0, -1]
        else:
            xgeo = geo_from_base(xb, mb)
            probs = net_new(xgeo).softmax(-1)[0, -1]
    conf, cls = probs.max(0)
    return int(cls.item()), float(conf.item())

def put_label(img, text, x, y, bg, scale=0.75):
    (w, h), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
    cv2.rectangle(img, (x, y-h-base-6), (x+w+10, y+2), bg, -1)
    cv2.putText(img, text, (x+5, y-base), cv2.FONT_HERSHEY_SIMPLEX, scale, BLACK, 2, cv2.LINE_AA)

def draw_hand(img, landmarks, color, mirror=True):
    h, w = img.shape[:2]
    px, py = (landmarks[:, 0] * w).astype(int), (landmarks[:, 1] * h).astype(int)
    if mirror:
        px = w - 1 - px
    for a, b in BONES:
        cv2.line(img, (px[a], py[a]), (px[b], py[b]), color, 2)
    cv2.rectangle(img, (max(0, px.min()-12), max(0, py.min()-12)),
                  (min(w-1, px.max()+12), min(h-1, py.max()+12)), color, 2)

def draw_hud(img, fps, threshold, which):
    put_label(img, f"FPS {fps:.1f} | threshold {threshold:.2f}", 10, 30, (230, 230, 230), 0.6)
    put_label(img, f"MODEL: {'BASELINE' if which == 'old' else 'TASK 8 STEP 2'}", 10, 65, (230, 230, 230), 0.6)
    put_label(img, "m: doi model | [ ]: nguong | q: thoat", 10, 100, (230, 230, 230), 0.55)

print("Đã nạp MediaPipe; webcam chỉ chạy ở cell 8.")

Đã nạp MediaPipe; webcam chỉ chạy ở cell 8.


In [25]:
#cell 8: chạy webcam; nhấn m để đổi model, [ ] chỉnh ngưỡng, q để thoát
CAM_ID, MIRROR, THR_CAM, WHICH = 0, True, 0.70, "old"
cap = cv2.VideoCapture(CAM_ID, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 30)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
assert cap.isOpened(), "Không mở được webcam; thử đổi CAM_ID = 1."

buf.clear()
state.update(prev_lm=None, prev_t=None)
fps, last = 0.0, time.perf_counter()
try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        now = time.perf_counter()
        lm = detect(frame, int((now - T0) * 1000))
        push(lm, now)
        out = predict(WHICH) if lm is not None else None
        disp = cv2.flip(frame, 1) if MIRROR else frame
        fps = 0.9 * fps + 0.1 / max(now - last, 1e-6)
        last = now

        if out is None:
            put_label(disp, "NO HAND", 10, 150, GRAY)
        else:
            cls, conf = out
            sure = conf >= THR_CAM
            color = COLORS[cls] if sure else GRAY
            draw_hand(disp, lm, color, MIRROR)
            put_label(disp, f"{CLASSES[cls]} {conf:.0%}" if sure else f"{CLASSES[cls]}? {conf:.0%}",
                      10, 150, color)
        draw_hud(disp, fps, THR_CAM, WHICH)
        cv2.imshow("Task 8 Step 2 - RPS webcam", disp)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key == ord("m"):
            WHICH = "new" if WHICH == "old" else "old"
        elif key == ord("["):
            THR_CAM = max(0.05, THR_CAM - 0.05)
        elif key == ord("]"):
            THR_CAM = min(0.95, THR_CAM + 0.05)
finally:
    cap.release()
    cv2.destroyAllWindows()
    landmarker.close()
print("Đã đóng webcam.")

Đã đóng webcam.


## Cách quyết định giữ hay bỏ Step 2

- So với `wide45_lownoise`, ưu tiên **không giảm quá 1 điểm phần trăm ở góc 0°** và **tăng độ chính xác ở ±45°**.
- Xem thêm độ chính xác khi confidence ≥ 0.7 và độ phủ; model không nên chỉ tự tin trên một số ít khung.
- Sau đó dùng cell 8, nhấn `m` để đổi model ngay khi giữ cùng một cử chỉ và thử nhiều góc. Nếu không thấy cải thiện thực tế thì giữ baseline.
- Ghi lại: Δ 0°, Δ ±45°, confidence/coverage và nhận xét webcam vào `Decision(5).docx`.

Lưu ý: khoảng cách 3D có thể giảm độ nhạy với phép xoay toán học, nhưng MediaPipe ước lượng tọa độ z có nhiễu. Vì vậy đây là giả thuyết cần kiểm tra bằng số liệu và webcam, không mặc định rằng model mới tốt hơn.